### TRANSFORMATION WORKFLOW

1. RENAME COLUMNS
2. TRIM SPACES
3. CLEAN CUSTOMER_ID (remove "NAS" prefix)
4. HANDLE NULL CUSTOMER_GENDER
5. WRITE INTO SILVER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
# 0) LOAD DATA & READ FROM BRONZE
df_erpc = spark.table("acdproj.bronze.erp_cust_az12")

In [0]:
# 1) RENAME COLUMNS (early, so the rest of the code reads clearly)
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "customer_gender"
}
for old_name, new_name in RENAME_MAP.items():
    df_erpc = df_erpc.withColumnRenamed(old_name, new_name)

In [0]:
# 2) TRIM SPACES
for field in df_erpc.schema.fields:
    if isinstance(field.dataType, StringType):
        df_erpc = df_erpc.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
# 3) CLEAN CUSTOMER_ID (remove "NAS" prefix to align with crm_cust_info.customer_key)
df_erpc = df_erpc.withColumn("customer_id", F.substring(F.col("customer_id"), 4, 100))

In [0]:
# 4) NORMALIZE CUSTOMER_GENDER
# Source mixes abbreviated (M/F) and full (Male/Female) values, plus nulls

df_erpc = df_erpc.withColumn(
    "customer_gender",
    F.when(F.upper(F.col("customer_gender")).isin("M", "MALE"), "Male")
     .when(F.upper(F.col("customer_gender")).isin("F", "FEMALE"), "Female")
     .otherwise("n/a")
)

In [0]:
# 5) WRITE INTO SILVER
df_erpc.write.mode("overwrite").saveAsTable("acdproj.silver.erp_customers")